# prep

In [1]:
import os

folder_path = "/Users/yerik/Music/try_new mp3/_1_YOGA"

file_names = [
    f for f in os.listdir(folder_path)
    if os.path.isfile(os.path.join(folder_path, f))
]

print(file_names)


['YOGA-7_DeepDub.txt', 'YOGA-9_ECSA-ARAB.txt', 'YOGA-3_Viby-FLOWS.txt', '.DS_Store', 'YOGA-2_Melo-no-words.txt', 'YOGA-10_ECSA-afronhouse.txt', 'YOGA-8_ecsa-remix.txt', 'YOGA-6_Bass-d&B.txt', 'YOGA-5_Chamanics.txt', 'YOGA-12_ECSA-FunkyStart.txt', 'YOGA-11_ECSA-hastaARRIBA.txt', 'YOGA-1_Meditate_astral.txt', 'YOGA-4_Bilateral_fxs.txt']


# copy files to new folder - from txt RK from each element in list 


In [1]:
# ============================================================
# === TXT (BOM DETECT) → AUTO FOLDER COPY → SAFE REWRITE TXT ==
# ============================================================

import os
import shutil
import pandas as pd
from tqdm import tqdm


def _copy_0701_txt_replace_SAFE_GET_df(txt_path,
                                      path_col='Location',
                                      sep='\t',
                                      overwrite_files=True,
                                      overwrite_txt=True):
    if not os.path.isfile(txt_path):
        raise FileNotFoundError(f"TXT not found: {txt_path}")

    # -------- BOM detect encoding --------
    with open(txt_path, 'rb') as f:
        head = f.read(4)

    if head.startswith(b'\xff\xfe') or head.startswith(b'\xfe\xff'):
        enc = 'utf-16'
    elif head.startswith(b'\xef\xbb\xbf'):
        enc = 'utf-8-sig'
    else:
        enc = None

    # -------- read txt --------
    enc_try = ([enc] if enc else []) + ['utf-16', 'utf-16-le', 'utf-8-sig', 'utf-8', 'latin-1']
    df = None
    used_enc = None
    read_errors = []

    for e in enc_try:
        if e is None:
            continue
        try:
            df = pd.read_csv(txt_path, sep=sep, encoding=e, dtype=str, engine='python')
            used_enc = e
            break
        except Exception as ex:
            read_errors.append((e, str(ex)))

    if df is None:
        raise UnicodeError(f"Failed reading TXT. Tried encodings: {read_errors}")

    df.columns = df.columns.str.strip()

    if path_col not in df.columns:
        raise ValueError(f"Column '{path_col}' not found. Columns: {list(df.columns)}")

    # -------- auto destination folder --------
    txt_dir = os.path.dirname(txt_path)
    txt_base = os.path.splitext(os.path.basename(txt_path))[0]
    dest_folder = os.path.join(txt_dir, txt_base)
    os.makedirs(dest_folder, exist_ok=True)

    # -------- sanitize paths --------
    def _clean_path(p):
        if p is None:
            return ""
        p = str(p)
        return p.replace('\ufeff', '').replace('\r', '').strip().strip('"').strip("'")

    # preserve original paths ALWAYS (prevents self-sabotage on second run)
    if 'Original_Location' not in df.columns:
        df['Original_Location'] = df[path_col]

    src_list = df['Original_Location'].fillna('').astype(str).tolist()

    new_paths = []
    status = []
    errors = []

    print(f"\nTXT  → {txt_path}")
    print(f"ENC  → {used_enc}")
    print(f"DEST → {dest_folder}")
    print(f"ROWS → {len(df)}\n")

    for src in tqdm(src_list, desc='Copying files', unit='file'):
        try:
            src = _clean_path(src)

            if (not src) or (not os.path.isfile(os.path.expanduser(src))):
                new_paths.append(None)
                status.append('missing')
                errors.append('not found')
                continue

            src2 = os.path.expanduser(src)
            fname = os.path.basename(src2)
            dst = os.path.join(dest_folder, fname)

            if overwrite_files or (not os.path.exists(dst)):
                shutil.copy2(src2, dst)

            new_paths.append(dst)
            status.append('copied')
            errors.append('')

        except Exception as e:
            new_paths.append(None)
            status.append('error')
            errors.append(str(e))

    # Only update Location when we actually copied successfully
    df['copy_status'] = status
    df['copy_error'] = errors

    df[path_col] = [
        new_paths[i] if status[i] == 'copied' else _clean_path(df.loc[i, 'Original_Location'])
        for i in range(len(df))
    ]

    # -------- overwrite txt in place --------
    if overwrite_txt:
        tmp_path = txt_path + ".__tmp__"
        out_enc = used_enc if used_enc else 'utf-16'

        df.to_csv(tmp_path, sep=sep, index=False, encoding=out_enc, lineterminator='\n')
        os.replace(tmp_path, txt_path)
        print(f"\nUPDATED TXT OVERWRITTEN → {txt_path}\n")

    return df


In [2]:
import os

#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

TXT_FOLDER = "/Users/yerik/Music/try_new mp3/_1_YOGA"

In [3]:

#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

txt_paths = [
    os.path.join(TXT_FOLDER, f)
    for f in os.listdir(TXT_FOLDER)
    if f.endswith('.txt') and os.path.isfile(os.path.join(TXT_FOLDER, f))
]

for TXT_PATH in txt_paths:
    print(f'RUNNING → {TXT_PATH}')

    df = _copy_0701_txt_replace_SAFE_GET_df(
        txt_path=TXT_PATH,
        path_col='Location',
        sep='\t',
        overwrite_files=True,
        overwrite_txt=True
    )


RUNNING → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-7_DeepDub.txt

TXT  → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-7_DeepDub.txt
ENC  → utf-16
DEST → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-7_DeepDub
ROWS → 14



Copying files: 100%|████████████████████████████████████████████████████████| 14/14 [00:00<00:00, 57.97file/s]



UPDATED TXT OVERWRITTEN → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-7_DeepDub.txt

RUNNING → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-9_ECSA-ARAB.txt

TXT  → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-9_ECSA-ARAB.txt
ENC  → utf-16
DEST → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-9_ECSA-ARAB
ROWS → 16



Copying files: 100%|████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 73.53file/s]



UPDATED TXT OVERWRITTEN → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-9_ECSA-ARAB.txt

RUNNING → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-3_Viby-FLOWS.txt

TXT  → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-3_Viby-FLOWS.txt
ENC  → utf-16
DEST → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-3_Viby-FLOWS
ROWS → 19



...


UPDATED TXT OVERWRITTEN → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-11_ECSA-hastaARRIBA.txt

RUNNING → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-1_Meditate_astral.txt

TXT  → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-1_Meditate_astral.txt
ENC  → utf-16
DEST → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-1_Meditate_astral
ROWS → 10



Copying files: 100%|████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 89.64file/s]



UPDATED TXT OVERWRITTEN → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-1_Meditate_astral.txt

RUNNING → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-4_Bilateral_fxs.txt

TXT  → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-4_Bilateral_fxs.txt
ENC  → utf-16
DEST → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-4_Bilateral_fxs
ROWS → 16



Copying files: 100%|████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 21.52file/s]


UPDATED TXT OVERWRITTEN → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-4_Bilateral_fxs.txt



# rename ALL FILES ACCORDING TO THE CODE WORD AND TXTS TAHT HAVE BEEN UPTADED NEW PATHS 

In [4]:
# ============================================================
# ====== FIX LOCATION PATHS (BOM/CR/QUOTES) + RENAME ==========
# ============================================================

import os
import pandas as pd
from tqdm import tqdm
from datetime import datetime


def _rename_0701_fixpaths_GET_df_from_txt(
    txt_path,
    custom_artist="DJ_Selphi",
    custom_genre="Salsa",
    custom_label="Bachata",
    custom_release_date="",
    custom_purchase_date="",
    sep="\t"
):
    tqdm.pandas()

    # --- BOM detect read (same logic as before, no guessing) ---
    with open(txt_path, "rb") as f:
        head = f.read(4)

    if head.startswith(b"\xff\xfe") or head.startswith(b"\xfe\xff"):
        enc = "utf-16"
    elif head.startswith(b"\xef\xbb\xbf"):
        enc = "utf-8-sig"
    else:
        enc = "utf-16"  # most DJ exports

    df = pd.read_csv(
        txt_path,
        sep=sep,
        encoding=enc,
        engine="python",
        on_bad_lines="skip",
        dtype=str
    )

    df.columns = df.columns.str.strip()

    # ---------- helpers ----------
    def _clean_str(s):
        if s is None:
            return ""
        s = str(s)
        # strip BOM + whitespace + CR
        return s.replace("\ufeff", "").replace("\r", "").strip().strip('"').strip("'")

    def _clean_filename_token(s):
        return (
            _clean_str(s)
            .replace(" ", "_").replace("/", "___").replace(",", "_")
            .replace("(", "").replace(")", "").replace("!", "")
            .replace("&", "and").replace("’", "").replace("'", "")
            .replace("¿", "").replace("¡", "").replace(":", "")
            .replace(";", "").strip()
        )

    def extract_mix(title):
        t = _clean_str(title).lower()
        if ("remix" in t) or ("mix" in t):
            return _clean_filename_token(title)
        return "original"

    def _parse_date_yyyymmdd(val):
        dt = pd.to_datetime(_clean_str(val), errors="coerce")
        return dt.strftime("%Y_%m_%d") if pd.notna(dt) else "NA"

    def _resolve_existing_path(p):
        """
        1) Try as-is
        2) Try expanding ~
        3) If still missing, try locating by filename in same folder as TXT’s auto-folder
           (this helps if Location column is stale but files were copied)
        """
        p = _clean_str(p)
        if not p:
            return None

        p2 = os.path.expanduser(p)
        if os.path.isfile(p2):
            return p2

        # fallback: try find by basename in the auto-folder created from TXT name
        txt_dir = os.path.dirname(txt_path)
        txt_base = os.path.splitext(os.path.basename(txt_path))[0]
        autofolder = os.path.join(txt_dir, txt_base)

        fname = os.path.basename(p2)
        cand = os.path.join(autofolder, fname)
        if os.path.isfile(cand):
            return cand

        return None

    def format_filename(row):
        title = _clean_filename_token(row.get("Track Title", ""))[:25]
        remix = extract_mix(row.get("Track Title", ""))

        artist_val = _clean_filename_token(custom_artist)[:25] if custom_artist else _clean_filename_token(row.get("Artist", ""))[:25]
        genre_val = _clean_filename_token(custom_genre) if custom_genre else _clean_filename_token(row.get("Genre", ""))
        label_val = _clean_filename_token(custom_label) if custom_label else _clean_filename_token(row.get("Label", ""))

        release_date_val = custom_release_date if custom_release_date else _parse_date_yyyymmdd(row.get("Release Date", ""))
        key = _clean_filename_token(row.get("Key", "NA")) or "NA"

        bpm_raw = _clean_str(row.get("BPM", ""))
        try:
            bpm = str(int(round(float(bpm_raw))))
        except Exception:
            bpm = "NA"

        purchase_date_val = custom_purchase_date if custom_purchase_date else _parse_date_yyyymmdd(row.get("Date Added", datetime.today()))

        # extension comes from Location after resolution (safer), but fallback to .mp3
        return title, artist_val, remix, key, bpm, genre_val, label_val, release_date_val, purchase_date_val

    # ---------- main loop ----------
    new_paths = []
    missing_rows = 0

    for i, row in tqdm(df.iterrows(), total=len(df), desc="Renaming files"):
        original_path_raw = row.get("Location", "")
        original_path = _resolve_existing_path(original_path_raw)

        if not original_path:
            missing_rows += 1
            new_paths.append(None)
            continue

        title, artist_val, remix, key, bpm, genre_val, label_val, release_date_val, purchase_date_val = format_filename(row)
        ext = os.path.splitext(original_path)[1] or ".mp3"

        new_filename = (
            f"TRkw_{title}_ARkw_{artist_val}_MXkw_{remix}_KYkw_{key}_"
            f"BPkw_{bpm}_GNkw_{genre_val}_LBkw_{label_val}_RYkw_{release_date_val}_"
            f"PYkw_{purchase_date_val}{ext}"
        )

        if len(new_filename) > 240:
            new_filename = new_filename[:230] + ext

        new_path = os.path.join(os.path.dirname(original_path), new_filename)

        try:
            os.rename(original_path, new_path)
            new_paths.append(new_path)
        except Exception:
            new_paths.append(None)

    df["Renamed_Path"] = new_paths

    if missing_rows:
        print(f"\n⚠️ Missing files for {missing_rows} rows.")
        print("Most likely: Location has stale paths, hidden characters, or copy_status != copied.\n")

    return df


In [5]:
import os

#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!


#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

txt_paths = [
    os.path.join(TXT_FOLDER, f)
    for f in os.listdir(TXT_FOLDER)
    if f.endswith('.txt') and os.path.isfile(os.path.join(TXT_FOLDER, f))  # same-level only
]

for TXT_PATH in txt_paths:
    print(f'RENAME → {TXT_PATH}')

    df = _rename_0701_fixpaths_GET_df_from_txt(
        txt_path=TXT_PATH,
        custom_artist="",
        custom_genre="",
        custom_label=""
    )


RENAME → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-7_DeepDub.txt


Renaming files: 100%|███████████████████████████████████████████████████████| 14/14 [00:00<00:00, 1861.18it/s]


RENAME → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-9_ECSA-ARAB.txt


Renaming files: 100%|███████████████████████████████████████████████████████| 16/16 [00:00<00:00, 2740.26it/s]


RENAME → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-3_Viby-FLOWS.txt


...

Renaming files: 100%|█████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 2399.88it/s]


RENAME → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-1_Meditate_astral.txt


Renaming files: 100%|███████████████████████████████████████████████████████| 10/10 [00:00<00:00, 2483.31it/s]


RENAME → /Users/yerik/Music/try_new mp3/_1_YOGA/YOGA-4_Bilateral_fxs.txt


Renaming files: 100%|███████████████████████████████████████████████████████| 16/16 [00:00<00:00, 2877.12it/s]


# Transform to AIFF


In [6]:
# -----######-----###### CORE IMPORTABLE FUNCTION (All → AIFF 44.1/16/Stereo + Tags + Verify) -----######-----###### #
import os, sys, shutil, subprocess, tempfile
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

# Tagging
from mutagen import File as MutaFile
from mutagen.aiff import AIFF
from mutagen.id3 import (
    ID3, ID3NoHeaderError, ID3BadUnsynchData,
    TIT2, TPE1, TPE2, TALB, TCON, TDRC, TRCK, TPOS, COMM, TBPM, TKEY,
    TPUB, TSRC, TPE3, TCOM, TENC, APIC, CTOC, CHAP
)

# -------------------- helpers (no ASCII banner for sub-fns) -------------------- #
def _safe_get_first(d, key):
    if d is None: return None
    v = d.get(key)
    if v is None: return None
    if isinstance(v, (list, tuple)):
        return v[0] if v else None
    return v

def _as_int_pair(text):
    if not text: return None, None
    s = str(text)
    if '/' in s:
        a,b = s.split('/',1)
        return (a.strip() or None), (b.strip() or None)
    return (s.strip() or None), None

def _ensure_id3(aiff_path):
    a = AIFF(aiff_path)
    if a.tags is None:
        a.add_tags()
    return a

def _copy_id3_frames(src_id3, dst_id3):
    # Copy common frames + chapters/artwork; ignore oddities that fail to serialize.
    for frame in list(src_id3.values()):
        try:
            if isinstance(frame, APIC):
                dst_id3.add(APIC(encoding=frame.encoding, mime=frame.mime, type=frame.type, desc=frame.desc, data=frame.data))
            elif isinstance(frame, COMM):
                dst_id3.add(COMM(encoding=frame.encoding, lang=frame.lang, desc=frame.desc, text=frame.text))
            elif isinstance(frame, (TIT2, TPE1, TPE2, TALB, TCON, TDRC, TRCK, TPOS, TBPM, TKEY, TPUB, TSRC, TPE3, TCOM, TENC)):
                dst_id3.add(type(frame)(encoding=frame.encoding, text=frame.text))
            elif isinstance(frame, (CTOC, CHAP)):
                dst_id3.add(frame)
            else:
                # Pass through for other safe T* frames
                dst_id3.add(frame)
        except Exception:
            # Skip non-serializable frames without killing the run
            pass

def _map_generic_to_id3(vtags, dst_id3, pictures=None):
    # Generic (Vorbis/FLAC/WAV INFO) → ID3
    title   = _safe_get_first(vtags, "title")
    artist  = _safe_get_first(vtags, "artist")
    album   = _safe_get_first(vtags, "album")
    albumartist = _safe_get_first(vtags, "albumartist") or _safe_get_first(vtags, "album artist")
    genre   = _safe_get_first(vtags, "genre")
    date    = _safe_get_first(vtags, "date") or _safe_get_first(vtags, "year")
    comment = _safe_get_first(vtags, "comment") or _safe_get_first(vtags, "description")
    bpm     = _safe_get_first(vtags, "bpm")
    key_    = _safe_get_first(vtags, "initialkey") or _safe_get_first(vtags, "key")
    label   = _safe_get_first(vtags, "label") or _safe_get_first(vtags, "publisher")
    isrc    = _safe_get_first(vtags, "isrc")
    remixer = _safe_get_first(vtags, "remixer")
    composer= _safe_get_first(vtags, "composer")
    encoder = _safe_get_first(vtags, "encoder") or _safe_get_first(vtags, "encodedby") or _safe_get_first(vtags, "encoded_by")
    trk     = _safe_get_first(vtags, "tracknumber")
    dsk     = _safe_get_first(vtags, "discnumber")

    if title:   dst_id3.add(TIT2(encoding=3, text=str(title)))
    if artist:  dst_id3.add(TPE1(encoding=3, text=str(artist)))
    if album:   dst_id3.add(TALB(encoding=3, text=str(album)))
    if albumartist: dst_id3.add(TPE2(encoding=3, text=str(albumartist)))
    if genre:   dst_id3.add(TCON(encoding=3, text=str(genre)))
    if date:    dst_id3.add(TDRC(encoding=3, text=str(date)))
    if comment: dst_id3.add(COMM(encoding=3, lang="eng", desc="", text=str(comment)))
    if bpm:     dst_id3.add(TBPM(encoding=3, text=str(bpm)))
    if key_:    dst_id3.add(TKEY(encoding=3, text=str(key_)))
    if label:   dst_id3.add(TPUB(encoding=3, text=str(label)))
    if isrc:    dst_id3.add(TSRC(encoding=3, text=str(isrc)))
    if remixer: dst_id3.add(TPE3(encoding=3, text=str(remixer)))
    if composer:dst_id3.add(TCOM(encoding=3, text=str(composer)))
    if encoder: dst_id3.add(TENC(encoding=3, text=str(encoder)))

    if trk:
        n, d = _as_int_pair(trk)
        if n or d:
            dst_id3.add(TRCK(encoding=3, text=[f"{n or ''}/{d or ''}".strip('/')]))
    if dsk:
        n, d = _as_int_pair(dsk)
        if n or d:
            dst_id3.add(TPOS(encoding=3, text=[f"{n or ''}/{d or ''}".strip('/')]))

    if pictures:
        for pic in pictures:
            try:
                dst_id3.add(APIC(encoding=3, mime=getattr(pic, "mime", None) or "image/jpeg", type=3, desc=u"", data=getattr(pic, "data", b"")))
            except Exception:
                pass

def _copy_all_tags_to_aiff(src_path, aiff_path):
    """
    After encoding, write a clean ID3 tag set into the AIFF.
    Priority:
      1) If source has ID3 → copy frames
      2) Else, map Vorbis/FLAC/WAV INFO → ID3; copy artwork when possible
    """
    dst_aiff = _ensure_id3(aiff_path)
    dst_id3 = dst_aiff.tags

    src = MutaFile(src_path)
    if src is None:
        dst_aiff.save()
        return

    # Direct ID3 → ID3
    try:
        src_id3 = getattr(src, "tags", None)
        if isinstance(src_id3, ID3) or (src_id3 and any(k.startswith("T") or k in ("APIC","COMM","CTOC","CHAP") for k in src_id3.keys())):
            try:
                _copy_id3_frames(src_id3, dst_id3)
                dst_aiff.save()
                return
            except Exception:
                pass
    except Exception:
        pass

    # Generic mapping (Vorbis/FLAC/WAV INFO, MP4 atoms won't map fully)
    vtags = getattr(src, "tags", {}) or {}
    pictures = []
    try:
        # FLAC: embedded pictures
        if hasattr(src, "pictures") and getattr(src, "pictures", None):
            pictures = src.pictures
        elif hasattr(src, "tags") and "METADATA_BLOCK_PICTURE" in src.tags:
            pictures = []  # base64 case skipped (mutagen handles some variants)
    except Exception:
        pictures = []

    try:
        _map_generic_to_id3(vtags, dst_id3, pictures=pictures)
    except Exception:
        pass

    dst_aiff.save()

def _verify_aiff_ok(aiff_path):
    try:
        t = AIFF(aiff_path)
        _ = t.info.length  # raises if broken
        return True, None
    except Exception as e:
        return False, str(e)

def _exts_casefold(exts):
    # normalize to a case-insensitive set, include upper/lower/Title variants
    s = set()
    for e in exts or []:
        if not e: continue
        ee = e if e.startswith(".") else "."+e
        base = ee.lower()
        s.add(base)
        s.add(base.upper())
        s.add(base.capitalize())
    return s

def _ffmpeg_encode_to_aiff(src_path, dst_path):
    """
    Robust ffmpeg call:
    - force AIFF PCM 16-bit big-endian, 44.1kHz, stereo
    - strip container metadata (we'll write fresh ID3 next)
    - disable video/subs
    - choose first audio stream explicitly
    """
    cmd = [
        "ffmpeg",
        "-hide_banner", "-loglevel", "error",
        "-y",
        "-i", str(src_path),
        "-map", "0:a:0",
        "-vn", "-sn",
        "-ar", "44100",
        "-ac", "2",
        "-c:a", "pcm_s16be",
        "-map_metadata", "-1",
        str(dst_path)
    ]
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return p.returncode == 0

# -----######-----###### CORE IMPORTABLE FUNCTION (per-file) -----######-----###### #
def _aiff_0109_onefile_GET_status_path(
    file_in,
    out_dir=None
):
    """
    Convert a single file → AIFF (44.1kHz / 16-bit / stereo), always re-encode (even AIFF).
    Returns (status, path_out). status in {"ok", "broken", "fail"}.
    """
    src_path = Path(file_in)
    if out_dir is None:
        out_dir = src_path.parent
    else:
        out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)

    dst_path = out_dir / (src_path.stem + ".aiff")

    # Atomic write via temp file to avoid half-written outputs
    with tempfile.TemporaryDirectory() as td:
        tmp_path = Path(td) / (src_path.stem + ".aiff")

        ok = _ffmpeg_encode_to_aiff(src_path, tmp_path)
        if not ok or not tmp_path.exists():
            return "fail", None

        # Write tags/artwork (best-effort; never fatal)
        try:
            _copy_all_tags_to_aiff(src_path, tmp_path)
        except Exception:
            pass

        # Verify playable
        v_ok, v_err = _verify_aiff_ok(tmp_path)
        if not v_ok:
            # Move the broken file out with suffix
            broken_dst = dst_path.with_stem(dst_path.stem + "_BROKEN")
            try:
                broken_dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(tmp_path), str(broken_dst))
            except Exception:
                pass
            return "broken", broken_dst if broken_dst.exists() else dst_path

        # All good: move into place
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(tmp_path), str(dst_path))

    return "ok", dst_path

# -----######-----###### CORE IMPORTABLE FUNCTION (batch) -----######-----###### #
def _aiff_0109_all2aiff_GET_summary(
    root_folder,
    out_root=None,
    audio_extensions=None,
    dry_run="n",
    src_action="keep",          # "keep" | "move" | "trash"
    src_action_move_dir=None,   # required if src_action == "move"
    overwrite="n"               # "y" to overwrite existing AIFF at dst, else skip creating duplicate
):
    """
    Recursively convert *major* audio types to AIFF (CDJ-safe 44.1/16/stereo) with tags & artwork.
    - Always re-encodes, even for AIFF sources (to guarantee target spec)
    - Mirrors folder structure under out_root (if provided)
    - Case-insensitive extension handling
    - macOS junk skipped
    - TQM progress bar + compact summary
    - Post-success actions on sources: keep | move | trash

    Returns:
      summary dict with counts and lists per status.
    """
    root = Path(root_folder)

    # If user doesn't pass, we include a broad set of common types
    if audio_extensions is None:
        audio_extensions = [
            ".flac", ".wav", ".mp3", ".aiff", ".aif",
            ".m4a", ".aac", ".alac", ".ogg", ".oga", ".wv", ".aifc"
        ]
    exts_all = _exts_casefold(audio_extensions)

    # Collect candidates (skip macOS junk)
    all_files = [
        p for p in root.rglob("*")
        if p.is_file()
        and not p.name.startswith("._")
        and p.name != ".DS_Store"
        and (p.suffix in exts_all)
    ]

    # Compute outputs + decide skips
    targets = []
    for src_path in all_files:
        rel = src_path.relative_to(root)
        out_dir = (Path(out_root) / rel.parent) if out_root else src_path.parent
        dst_path = out_dir / (src_path.stem + ".aiff")

        if dry_run.lower().startswith("y"):
            targets.append((src_path, out_dir, dst_path, "todo"))
        else:
            # If overwrite == 'n' and AIFF already exists in destination, skip re-creating file
            if dst_path.exists() and not overwrite.lower().startswith("y"):
                # Still *verify* later if the existing AIFF matches spec? We assume OK for speed.
                # If you want forced re-encode, set overwrite='y'.
                continue
            targets.append((src_path, out_dir, dst_path, "todo"))

    # ----- TQM BAR -----
    pbar = tqdm(total=len(targets), desc="TQM | All → AIFF 44.1/16/stereo", unit="file")

    stats = {
        "ok": 0, "broken": 0, "fail": 0,
        "skipped_existing": 0,
        "total_scanned": len(all_files),
        "total_planned": len(targets),
        "ok_paths": [], "broken_paths": [], "fail_paths": [], "skipped_paths": []
    }

    # If dry-run: just preview names and return summary
    if dry_run.lower().startswith("y"):
        for (src_path, out_dir, dst_path, _) in targets:
            pbar.set_postfix_str(f"DRY-RUN → {src_path.name}")
            pbar.update(1)
            stats["skipped_paths"].append(str(src_path))
        pbar.close()
        return stats

    # Convert
    for (src_path, out_dir, dst_path, _) in targets:
        # If we got here and the destination exists but overwrite == 'n', mark skipped
        if dst_path.exists() and not overwrite.lower().startswith("y"):
            stats["skipped_existing"] += 1
            stats["skipped_paths"].append(str(dst_path))
            pbar.set_postfix_str(f"SKIP (exists): {dst_path.name}")
            pbar.update(1)
            continue

        status, outp = _aiff_0109_onefile_GET_status_path(src_path, out_dir=out_dir)
        if status == "ok":
            stats["ok"] += 1
            stats["ok_paths"].append(str(outp))
            pbar.set_postfix_str(f"OK: {src_path.name}")
            # Post-success action on source
            try:
                if src_action == "trash":
                    # move to user Trash if available; fallback to unlink
                    try:
                        from send2trash import send2trash
                        send2trash(str(src_path))
                    except Exception:
                        src_path.unlink(missing_ok=True)
                elif src_action == "move":
                    if not src_action_move_dir:
                        raise ValueError("src_action_move_dir is required when src_action='move'")
                    dst_dir_move = Path(src_action_move_dir); dst_dir_move.mkdir(parents=True, exist_ok=True)
                    shutil.move(str(src_path), str(dst_dir_move / src_path.name))
                else:
                    pass  # keep
            except Exception:
                # Non-fatal; keep going
                pass

        elif status == "broken":
            stats["broken"] += 1
            stats["broken_paths"].append(str(outp))
            pbar.set_postfix_str(f"BROKEN: {src_path.name}")
        else:
            stats["fail"] += 1
            stats["fail_paths"].append(str(src_path))
            pbar.set_postfix_str(f"FAIL: {src_path.name}")

        pbar.update(1)

    pbar.close()

    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(
        f"[{stamp}] === SUMMARY ===\n"
        f"Total scanned:     {stats['total_scanned']}\n"
        f"Planned to convert:{stats['total_planned']}\n"
        f"Converted OK:      {stats['ok']}\n"
        f"Broken (tagged):   {stats['broken']}\n"
        f"Failed:            {stats['fail']}\n"
        f"Skipped (exists):  {stats['skipped_existing']}\n"
    )
    return stats


In [7]:
# ! CHANGE THESE AS NEEDED
root_folder = TXT_FOLDER


In [8]:
out_root    = None  # or e.g. "/Volumes/HD_back_UP/ALL_MUSIC/_AIFF_OUT"

# Externalized extensions (case-insensitive will be auto-handled)
audio_extensions = [".flac", ".wav", ".mp3", ".aiff", ".aif", ".m4a", ".aac", ".alac", ".ogg", ".oga", ".wv", ".aifc"]

# Action knobs
dry_run     = "n"           # "y" to preview only
overwrite   = "n"           # "y" to force re-encode even if dst exists
src_action  = "keep"        # "keep" | "move" | "trash"
move_dir    = ""  # required if src_action=="move"

summary = _aiff_0109_all2aiff_GET_summary(
    root_folder=root_folder,
    out_root=out_root,
    audio_extensions=audio_extensions,
    dry_run=dry_run,
    src_action=src_action,
    src_action_move_dir=move_dir,
    overwrite=overwrite
)


TQM | All → AIFF 44.1/16/stereo: 100%|█| 187/187 [00:51<00:00,  3.64file/s, OK: TRkw_INZO_-_Overthinker_myfree

[2026-01-08 19:37:37] === SUMMARY ===
Total scanned:     188
Planned to convert:187
Converted OK:      187
Broken (tagged):   0
Failed:            0
Skipped (exists):  0



# erase originals after checking 

In [9]:
# -----######-----###### CORE IMPORTABLE FUNCTION (Erase everything but AIFF) -----######-----###### #
import os
from pathlib import Path
from tqdm import tqdm

def _cleanup_2208_keepaiff_GET_removed_files(root_folder, dry_run="n"):
    """
    Recursively erase everything but AIFF (.aiff/.aif) files.
    Skips system junk (.DS_Store, ._*).
    
    Inputs:
      root_folder : str/Path → folder to clean
      dry_run     : "y" = preview only, "n" = actually delete
    
    Returns:
      dict summary with counts
    """
    root = Path(root_folder)

    # Collect all files
    all_files = [p for p in root.rglob("*") if p.is_file()]
    # Keep only those NOT AIFF
    targets = [
        p for p in all_files
        if p.suffix.lower() not in (".aiff", ".aif")
        and not p.name.startswith("._")
        and p.name != ".DS_Store"
    ]

    # Progress bar
    pbar = tqdm(total=len(targets), desc="TQM • Cleaning non-AIFF files", unit="file")

    removed, skipped = 0, 0
    for f in targets:
        if dry_run.lower().startswith("y"):
            pbar.set_postfix_str(f"DRY-RUN: would remove {f.name}")
            skipped += 1
        else:
            try:
                f.unlink()
                removed += 1
                pbar.set_postfix_str(f"Removed {f.name}")
            except Exception as e:
                skipped += 1
                pbar.set_postfix_str(f"⚠️ Skip {f.name}: {e}")
        pbar.update(1)

    pbar.close()

    summary = {
        "total_files": len(all_files),
        "removed": removed,
        "skipped": skipped,
        "kept_aiff": len(all_files) - len(targets),
    }

    print(
        f"Done cleanup.\n"
        f" • Total files scanned: {summary['total_files']}\n"
        f" • Removed: {summary['removed']}\n"
        f" • Skipped (errors/dry-run): {summary['skipped']}\n"
        f" • Kept AIFF: {summary['kept_aiff']}\n"
    )
    return summary


In [10]:
_cleanup_2208_keepaiff_GET_removed_files(root_folder, dry_run="n")


TQM • Cleaning non-AIFF files: 100%|█| 200/200 [00:00<00:00, 1473.09file/s, Removed TRkw_INZO_-_Overthinker_my

Done cleanup.
 • Total files scanned: 389
 • Removed: 200
 • Skipped (errors/dry-run): 0
 • Kept AIFF: 189



{'total_files': 389, 'removed': 200, 'skipped': 0, 'kept_aiff': 189}

# EMBED IMAGE COVER 

In [11]:
# ============================================================#
# 0_FNS
# ============================================================#

import os
import shutil
import tempfile
import hashlib
import pandas as pd
from tqdm import tqdm

from mutagen.aiff import AIFF
from mutagen.id3 import ID3, APIC, ID3NoHeaderError


# ----------------------------######----------------------------#
#   _art_0801_aiffpng_GET_df_embedreport_STRONG                 #
# ----------------------------######----------------------------#
def _art_0801_aiffpng_GET_df_embedreport_STRONG(png_path,
                                               folder_path,
                                               recursive=True,
                                               overwrite_apic=True,
                                               atomic_write=True,
                                               make_backup=True,
                                               strict_verify=True,
                                               dry_run=False,
                                               verbose=True):
    """
    STRONG / pipeline-safe PNG embedding into AIFF (AIF/AIFF).

    Key upgrades vs "basic":
    - Atomic writes (write to temp, then replace) if atomic_write=True
    - Optional backups
    - Defensive tag creation & ID3 loading
    - Removes all APIC frames when overwrite_apic=True
    - Strong verification (reload file, check APIC exists, bytes match, md5 match)
    - Returns a df report for auditing

    Notes:
    - AIFF cover art lives in ID3 APIC frames.
    - We store APIC: mime=image/png, type=3 (front cover), encoding=utf-8 (3), desc=""
    """

    # ------------------ VALIDATE INPUTS ------------------ #
    if not os.path.isfile(png_path):
        raise FileNotFoundError(f"PNG not found: {png_path}")
    if not os.path.isdir(folder_path):
        raise NotADirectoryError(f"Folder not found: {folder_path}")
    if not png_path.lower().endswith(".png"):
        raise ValueError("png_path must point to a .png file.")

    with open(png_path, "rb") as f:
        png_bytes = f.read()

    apic_bytes_expected = len(png_bytes)
    md5_png = hashlib.md5(png_bytes).hexdigest()

    # ------------------ COLLECT AIFF FILES ------------------ #
    def _is_bad_name(name):
        return name.startswith("._") or name.startswith(".DS") or name in [".DS_Store"]

    exts = {".aif", ".aiff"}
    paths = []

    if recursive:
        for root, _, files in os.walk(folder_path):
            for fn in files:
                if _is_bad_name(fn):
                    continue
                if os.path.splitext(fn)[1].lower() in exts:
                    paths.append(os.path.join(root, fn))
    else:
        for fn in os.listdir(folder_path):
            if _is_bad_name(fn):
                continue
            if os.path.splitext(fn)[1].lower() in exts:
                paths.append(os.path.join(folder_path, fn))

    paths = sorted(paths)

    if verbose:
        print(f"\nFound AIFF files: {len(paths)}")
        print(f"PNG bytes: {apic_bytes_expected} | md5: {md5_png}\n")

    # ------------------ HELPERS ------------------ #
    def _load_or_init_id3_for_aiff(aiff_obj, file_path):
        """
        Ensures we have an ID3 tag container we can write to.
        Handles: missing tags, weird internal tag state, no header, etc.
        """
        if aiff_obj.tags is None:
            # Try "official" mutagen add_tags() first
            try:
                aiff_obj.add_tags()
            except Exception:
                # Fall back to raw ID3
                try:
                    aiff_obj.tags = ID3(file_path)
                except ID3NoHeaderError:
                    aiff_obj.tags = ID3()
        else:
            # Sometimes tags exist but not properly ID3-like; try to ensure ID3 access works
            try:
                _ = aiff_obj.tags.getall("APIC")
            except Exception:
                try:
                    aiff_obj.tags = ID3(file_path)
                except ID3NoHeaderError:
                    aiff_obj.tags = ID3()

        return aiff_obj

    def _embed_png_id3(aiff_obj):
        """
        Embed PNG as APIC frame, optionally wiping existing APIC first.
        """
        if overwrite_apic:
            try:
                aiff_obj.tags.delall("APIC")
            except Exception:
                # If delall fails, brute-force by reinitializing tags then continue
                aiff_obj.tags = ID3()

        aiff_obj.tags.add(
            APIC(
                encoding=3,         # utf-8
                mime="image/png",
                type=3,             # front cover
                desc="",
                data=png_bytes
            )
        )
        return aiff_obj

    def _verify_apic(file_path):
        """
        Reload and confirm:
        - At least one APIC exists
        - One APIC matches expected byte length
        - md5 of APIC data matches the PNG md5
        """
        a = AIFF(file_path)

        apics = []
        if a.tags is not None:
            try:
                apics = list(a.tags.getall("APIC"))
            except Exception:
                apics = []

        # fallback: raw ID3 load
        if not apics:
            try:
                id3 = ID3(file_path)
                apics = list(id3.getall("APIC"))
            except Exception:
                apics = []

        if not apics:
            return False, None, None, "no_apic_found"

        # Look for a matching one
        for frame in apics:
            data = getattr(frame, "data", None)
            if not data:
                continue
            if len(data) == apic_bytes_expected and hashlib.md5(data).hexdigest() == md5_png:
                return True, len(data), hashlib.md5(data).hexdigest(), ""

        # If APIC exists but mismatch, return diagnostic for first usable frame
        for frame in apics:
            data = getattr(frame, "data", None)
            if data:
                return False, len(data), hashlib.md5(data).hexdigest(), "apic_mismatch"

        return False, None, None, "apic_no_data"

    # ------------------ PROCESS ------------------ #
    rows = []
    for p in tqdm(paths, desc="Embedding PNG into AIFF (STRONG)", total=len(paths)):
        file_name = os.path.basename(p)
        status = "ok"
        err = ""
        apic_bytes_written = None
        md5_written = None
        backup_path = None

        try:
            if dry_run:
                rows.append({
                    "Path": p,
                    "file_name": file_name,
                    "status": "dry_run",
                    "apic_bytes_expected": apic_bytes_expected,
                    "apic_bytes_written": None,
                    "md5_png": md5_png,
                    "md5_written": None,
                    "backup_path": None,
                    "error": ""
                })
                continue

            # Backup first (optional)
            if make_backup:
                backup_path = p + ".bak"
                if not os.path.exists(backup_path):
                    shutil.copy2(p, backup_path)

            # Atomic write path: operate on temp copy then replace
            target_path = p
            work_path = p

            if atomic_write:
                fd, tmp_path = tempfile.mkstemp(suffix=os.path.splitext(p)[1].lower())
                os.close(fd)
                shutil.copy2(p, tmp_path)
                work_path = tmp_path

            # Load / init tags
            audio = AIFF(work_path)
            audio = _load_or_init_id3_for_aiff(audio, work_path)

            # Embed
            audio = _embed_png_id3(audio)
            audio.save()

            # Replace original if atomic_write
            if atomic_write:
                shutil.copy2(work_path, target_path)
                try:
                    os.remove(work_path)
                except Exception:
                    pass

            # Verify
            ok, apic_bytes_written, md5_written, v_err = _verify_apic(target_path)

            if strict_verify and not ok:
                raise RuntimeError(f"verification_failed: {v_err} | bytes={apic_bytes_written} | md5={md5_written}")

        except Exception as e:
            status = "fail"
            err = str(e)

            # If we failed and we have a backup, restore it to keep pipeline safe
            if make_backup and backup_path and os.path.exists(backup_path):
                try:
                    shutil.copy2(backup_path, p)
                except Exception:
                    pass

        rows.append({
            "Path": p,
            "file_name": file_name,
            "status": status,
            "apic_bytes_expected": apic_bytes_expected,
            "apic_bytes_written": apic_bytes_written,
            "md5_png": md5_png,
            "md5_written": md5_written,
            "backup_path": backup_path,
            "error": err
        })

    df_report = pd.DataFrame(rows)

    if verbose:
        ok = (df_report["status"] == "ok").sum()
        fail = (df_report["status"] == "fail").sum()
        dr = (df_report["status"] == "dry_run").sum()
        print(f"\nDONE | ok={ok} | fail={fail} | dry_run={dr}\n")

        if fail > 0:
            print("Failures (top 15):")
            print(df_report.loc[df_report["status"] == "fail", ["file_name", "error"]].head(15).to_string(index=False))

    return df_report


In [13]:
# ============================================================#
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
# ============================================================#

png_path    = r"/Users/yerik/Music/try_new mp3/_1_YOGA/YOGA.png"
folder_path = TXT_FOLDER


df_report = _art_0801_aiffpng_GET_df_embedreport_STRONG(
    png_path=png_path,
    folder_path=folder_path,
    recursive=True,
    overwrite_apic=True,
    atomic_write=True,     # <- key for pipeline safety
    make_backup=True,      # <- creates .bak once (first run)
    strict_verify=True,    # <- fail if not PERFECT match
    dry_run=False,
    verbose=True
)




Found AIFF files: 188
PNG bytes: 764965 | md5: 8d1a0821b0449a4625e590ceb4973c9c



Embedding PNG into AIFF (STRONG): 100%|█████████████████████████████████████| 188/188 [00:19<00:00,  9.64it/s]


DONE | ok=188 | fail=0 | dry_run=0



# NORMALIZE 